In [1]:
import torch
from torch import Tensor
import torch.nn.functional as F
import transformer_lens
from transformer_lens import HookedTransformer, HookedTransformerConfig
from einops import einsum

from jaxtyping import Int, Float
from typing import List, Tuple, Optional, Literal
import numpy as np
from transformer_lens import utils
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [2]:
model = HookedTransformer.from_pretrained("gpt2-small")

Loaded pretrained model gpt2-small into HookedTransformer


In [3]:
dot = model.W_E @ model.W_E.T

In [20]:
import torch
from tqdm import tqdm

def find_near_zero_pairs_blockwise(M: torch.Tensor, eps: float = 0.1, block_size: int = 1000):
    assert M.ndim == 2 and M.shape[0] == M.shape[1], "M must be square"
    N = M.shape[0]

    results = []

    for start in tqdm(range(0, N, block_size), desc="Scanning blocks"):
        end = min(start + block_size, N)

        # only take the upper triangle portion
        block = M[start:end, start:]   # shape [block_size, N-start]
        rows, cols = torch.nonzero(block.abs() <= eps, as_tuple=True)

        for r, c in zip(rows.tolist(), cols.tolist()):
            i = start + r
            j = start + c
            if i < j:  # strictly upper triangle
                results.append((i, j, block[r, c].item()))

    return results


In [35]:
pairs = find_near_zero_pairs_blockwise(dot, eps=0.0001, block_size=500)
print(f"Found {len(pairs)} near-zero pairs")

Scanning blocks: 100%|███████████████████████████████████████| 101/101 [00:00<00:00, 2033.43it/s]

Found 10 near-zero pairs


In [36]:
from collections import defaultdict

def group_by_first(pairs):
    groups = defaultdict(list)
    for i, j, val in pairs:
        groups[i].append(j)
        
    return [group for group in groups.values()]

In [37]:
groups = group_by_first(pairs)
print(len(groups))

7


In [ ]:
# expected something. found nothing... maybe expectation was the problem.

In [43]:
[model.to_single_str_token(x) for x in groups[4]]

['usable', 'ueless']